# Material models from RefractiveIndex.INFO

> Build frequency- (and temperature-) dependent `meow` materials from published
> refractive-index data, either by fitting a **Sellmeier** model or by sampling
> a [RefractiveIndex.INFO](https://refractiveindex.info) entry directly.

`meow.ridb` has three layers:

1. **RefractiveIndex.INFO access** &mdash; parse a `refractiveindex.info` URL
   (the `?shelf=&book=&page=` form or a direct `.yml` link), fetch the entry's
   YAML (cached on disk), and evaluate its index whether it is given as
   *tabulated* `n`/`k` data or as a *formula* fit (Sellmeier, Cauchy, &hellip;).
2. **Sellmeier fitting** (`fit_sellmeier`) &mdash; fit an `N`-term model
   $n^2 = 1 + a_0 + \sum_j b_j\,\lambda^2/(\lambda^2 - c_j)$ over a chosen
   wavelength range of validity. `fit_temperature_sellmeier` extends this to
   data measured at several temperatures.
3. **meow material builders** &mdash; `material_from_ri` /
   `sellmeier_material` (isotropic `SampledMaterial`), `anisotropic_material`
   (a different entry/dataset per crystal axis), and `temperature_material`
   (a `(wavelength, temperature)`-dispersive material).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import meow as mw

### An offline-safe helper

Fetching from RefractiveIndex.INFO needs network access. So that this notebook
also runs offline, the helper below tries `mw.fetch_ri_entry` first and falls
back to a published coefficient set parsed with `mw.parse_ri_yaml`. The two
paths produce an identical `RIEntry`, so the rest of the notebook is unaffected.

In [ ]:
def entry_or_fallback(ref, *, formula_type, coefficients, wl_range, name):
    """Fetch an RI.info entry, or rebuild it from published coefficients."""
    try:
        entry = mw.fetch_ri_entry(ref, timeout=15)
        print(f"fetched {name!r} from refractiveindex.info")
    except Exception as exc:  # offline / moved entry -> use the published values
        print(f"offline fallback for {name!r} ({type(exc).__name__})")
        lo, hi = wl_range
        yaml = (
            f"DATA:\n"
            f"  - type: formula {formula_type}\n"
            f"    wavelength_range: {lo} {hi}\n"
            f"    coefficients: {coefficients}\n"
        )
        entry = mw.parse_ri_yaml(yaml, name=name)
    return entry

## 1. Fitting a Sellmeier model

We start from a published dispersion &mdash; fused silica (SiO$_2$, Malitson
1965, RefractiveIndex.INFO formula&nbsp;1) &mdash; and treat it as a set of
"measured" index points. `fit_sellmeier` recovers a compact `N`-term model over
a chosen **wavelength range of validity**, parameterized by the number of terms.

In [ ]:
silica = entry_or_fallback(
    "https://refractiveindex.info/?shelf=main&book=SiO2&page=Malitson",
    formula_type=1,
    coefficients="0 0.6961663 0.0684043 0.4079426 0.1162414 0.8974794 9.896161",
    wl_range=(0.21, 6.7),
    name="SiO2 (Malitson)",
)

# treat the published curve as measured data over a telecom-ish window
wls = np.linspace(0.4, 2.0, 60)
n_data = np.real(silica.index(wls))

# fit 1-, 2- and 3-term Sellmeier models over the same range of validity
fits = {n: mw.fit_sellmeier(wls, n_data, num_terms=n, wl_range=(0.4, 2.0))
        for n in (1, 2, 3)}
for n, model in fits.items():
    print(f"{n}-term fit: RMS index error = {model.rms_error:.2e}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(wls, n_data, "k.", ms=4, label="published (Malitson)")
fine = np.linspace(0.4, 2.0, 300)
for n, model in fits.items():
    ax1.plot(fine, model.index(fine), label=f"{n}-term Sellmeier")
ax1.set(xlabel="wavelength [um]", ylabel="refractive index n",
        title="Sellmeier fits to fused silica")
ax1.legend()

terms = list(fits)
rms = [fits[n].rms_error for n in terms]
ax2.semilogy(terms, rms, "o-")
ax2.set(xlabel="number of Sellmeier terms", ylabel="RMS index error",
        title="Fit quality vs. number of terms", xticks=terms)
ax2.grid(True, which="both", alpha=0.3)
fig.tight_layout()

Even a single term captures fused silica's gentle dispersion to $\sim10^{-3}$;
three terms recover the published curve to machine precision. Use `wl_range` to
trade terms for a wider validity window.

### From a fit to a `meow` material

`sellmeier_material` samples the analytic model onto a dense grid and returns a
dispersive `meow.SampledMaterial`, which evaluates at any `Environment`
wavelength.

In [ ]:
silica_mat = mw.sellmeier_material("SiO2", fits[3])
for wl in (0.5, 1.31, 1.55):
    n = silica_mat(mw.Environment(wl=wl, T=25.0))
    print(f"n(SiO2) @ {wl:.2f} um = {np.real(n):.5f}")

## 2. Pulling an entry from RefractiveIndex.INFO directly

`material_from_ri` builds a material straight from an `RIEntry` &mdash; no fit
needed when the published formula/table is already accurate. Pass `fit_terms` to
instead refit a compact Sellmeier model (handy to smooth noisy tabulated data or
to get an analytic model for export).

In [ ]:
# silicon (Salzberg & Villa 1957) over its transparency window
silicon = entry_or_fallback(
    "https://refractiveindex.info/?shelf=main&book=Si&page=Salzberg",
    formula_type=1,
    coefficients="0 10.6684293 0.301516485 0.0030434748 1.13475115 1.54133408 1104.0",
    wl_range=(1.357, 11.04),
    name="Si (Salzberg)",
)
si_mat = mw.material_from_ri(silicon, name="Si", wl_range=(1.4, 5.0), npts=200)

wls_si = np.linspace(1.4, 5.0, 200)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(wls_si, [np.real(si_mat(mw.Environment(wl=w, T=25.0))) for w in wls_si])
ax.set(xlabel="wavelength [um]", ylabel="refractive index n",
       title="Silicon SampledMaterial from RefractiveIndex.INFO")
ax.grid(True, alpha=0.3)
fig.tight_layout()
print("n(Si) @ 1.55 um =", float(np.real(si_mat(mw.Environment(wl=1.55, T=25.0)))))

## 3. Anisotropic materials: a different entry per axis

For a birefringent crystal you can feed a **different** RefractiveIndex.INFO
entry (or dataset) to each axis. Here lithium niobate (LiNbO$_3$, Zelmon 1997):
the ordinary index $n_o$ on the $x,y$ axes and the extraordinary index $n_e$ on
$z$ &mdash; a uniaxial crystal. `anisotropic_material` returns a
`SampledAnisotropicMaterial` with a full (diagonal) permittivity tensor.

In [ ]:
ln_o = entry_or_fallback(
    "main/LiNbO3/Zelmon-o", formula_type=2,
    coefficients="0 2.6734 0.01764 1.2290 0.05914 12.614 474.60",
    wl_range=(0.4, 5.0), name="LiNbO3 (ordinary)",
)
ln_e = entry_or_fallback(
    "main/LiNbO3/Zelmon-e", formula_type=2,
    coefficients="0 2.9804 0.02047 0.5981 0.0666 8.9543 416.08",
    wl_range=(0.4, 5.0), name="LiNbO3 (extraordinary)",
)

ln = mw.anisotropic_material(
    "LiNbO3", {"x": ln_o, "y": ln_o, "z": ln_e}, wl_range=(0.5, 4.0), npts=200
)
print("is_isotropic:", ln.is_isotropic)
eps = ln.eps_tensor(mw.Environment(wl=1.55, T=25.0))
no = np.sqrt(np.real(eps[0, 0]))
ne = np.sqrt(np.real(eps[2, 2]))
print(f"@1.55 um: n_o={no:.4f}, n_e={ne:.4f}, birefringence dn={ne - no:+.4f}")

In [ ]:
wls_ln = np.linspace(0.5, 4.0, 200)
no_w = np.real(ln_o.index(wls_ln))
ne_w = np.real(ln_e.index(wls_ln))
fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 4))
axa.plot(wls_ln, no_w, label="ordinary $n_o$ (x, y)")
axa.plot(wls_ln, ne_w, label="extraordinary $n_e$ (z)")
axa.set(xlabel="wavelength [um]", ylabel="refractive index",
        title="LiNbO$_3$ principal indices")
axa.legend()
axb.plot(wls_ln, ne_w - no_w, color="C3")
axb.axhline(0, color="k", lw=0.5)
axb.set(xlabel="wavelength [um]", ylabel="$n_e - n_o$",
        title="LiNbO$_3$ birefringence")
axb.grid(True, alpha=0.3)
fig.tight_layout()

## 4. Temperature-dependent Sellmeier models

When index data is available at several temperatures, `fit_temperature_sellmeier`
fits one Sellmeier model per temperature and then a polynomial-in-temperature for
each coefficient, so `n(lambda, T)` is continuous in both variables.

Here we synthesize multi-temperature data by applying a thermo-optic shift
($dn/dT$) to the fused-silica dispersion &mdash; in practice you would pass real
measured datasets, e.g. one RefractiveIndex.INFO entry per temperature.

In [ ]:
base = fits[3]            # the 3-term fused-silica model from section 1
dn_dT = 1.0e-5           # /degC (illustrative thermo-optic coefficient)
wls_T = np.linspace(0.5, 2.0, 60)
temps = [20.0, 40.0, 60.0, 80.0]
datasets = {T: (wls_T, base.index(wls_T) + dn_dT * (T - 20.0)) for T in temps}

tmodel = mw.fit_temperature_sellmeier(datasets, num_terms=3, t_degree=1)
print("fitted terms:", tmodel.num_terms, " t_ref:", tmodel.t_ref)
for T in (25.0, 75.0):
    n = float(tmodel.index(np.array([1.55]), T)[0])
    print(f"n @ 1.55 um, {T:.0f} C = {n:.6f}")

In [ ]:
fig, (axc, axd) = plt.subplots(1, 2, figsize=(11, 4))
for T in (20.0, 50.0, 80.0):
    axc.plot(wls_T, tmodel.index(wls_T, T), label=f"{T:.0f} C")
axc.set(xlabel="wavelength [um]", ylabel="refractive index n",
        title="n(lambda, T) from TemperatureSellmeier")
axc.legend()

Ts = np.linspace(20.0, 80.0, 40)
for wl in (0.6, 1.0, 1.55):
    axd.plot(Ts, [float(tmodel.index(np.array([wl]), T)[0]) for T in Ts],
             label=f"{wl:.2f} um")
axd.set(xlabel="temperature [C]", ylabel="refractive index n",
        title="Index vs. temperature")
axd.legend()
fig.tight_layout()

### A `(wavelength, temperature)`-dispersive `meow` material

`temperature_material` samples `n(lambda, T)` onto a grid and returns a
`SampledMaterial` whose `params` carry both `wl` and `T`; `meow` then
interpolates in the `Environment`'s wavelength **and** temperature.

In [ ]:
silica_T = mw.temperature_material("SiO2(T)", tmodel, wls=wls_T)
print("material params:", sorted(silica_T.params))
for T in (25.0, 50.0, 75.0):
    n = silica_T(mw.Environment(wl=1.55, T=T))
    print(f"n(SiO2) @ 1.55 um, {T:.0f} C = {np.real(n):.6f}")

## Summary

- `fit_sellmeier(wls, n, num_terms=..., wl_range=...)` &mdash; fit a Sellmeier
  model parameterized by term count and validity range.
- `fetch_ri_entry(url)` / `parse_ri_yaml(text)` &mdash; load a
  RefractiveIndex.INFO entry (tabulated or formula).
- `material_from_ri` / `sellmeier_material` &mdash; isotropic dispersive
  `SampledMaterial` (optionally via a Sellmeier refit with `fit_terms`).
- `anisotropic_material({"x": ..., "y": ..., "z": ...})` &mdash; per-axis
  entries for birefringent crystals.
- `fit_temperature_sellmeier` + `temperature_material` &mdash; continuous
  `n(lambda, T)` from multi-temperature datasets.